# BGE-M3 ile tam arşiv gömmesi

**Ne yapar:** `chunks.json` içindeki 14.532 chunk'ı BAAI/bge-m3 ile vektöre çevirir,
sonucu `BAAI_bge-m3.npy` olarak indirtir.

**Neden Colab:** yerel makinede GPU yok, CPU'da ölçülen hız saniyede 0,6 chunk —
yani ~8 saat. T4 GPU ile 15-25 dakika bekleniyor.

**Önce yapılacak:** Menüden `Runtime > Change runtime type > T4 GPU` seç.
GPU seçmeden çalıştırırsan hız yereldekiyle aynı olur, anlamı kalmaz.

**Kritik uyarı:** Bu notebook'taki gömme metni kurgusu (`gomulecek_metin`) yereldeki
`_scripts/embed_chunks.py` ile **birebir aynı olmak zorunda**. Farklı olursa sorgu
vektörleriyle belge vektörleri aynı uzayda olmaz ve skorlar sessizce bozulur.

In [ ]:
# 1) GPU var mı? Beklenen: 'Tesla T4' benzeri bir satır ve cuda: True.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('cuda:', torch.cuda.is_available())

In [ ]:
# 2) Kurulum. 1-2 dakika sürer, sonunda sürüm numarası yazar.
!pip install -q sentence-transformers
import sentence_transformers
print('sentence-transformers', sentence_transformers.__version__)

## chunks.json yükleme

İki yol var, birini seç:

- **A) Doğrudan yükleme** (aşağıdaki hücre): dosya seçme penceresi açılır.
  Dosya büyük olduğu için birkaç dakika sürebilir.
- **B) Google Drive**: dosyayı önce Drive'a koyduysan daha hızlı, ayrıca
  bağlantı koparsa yeniden yüklemek gerekmez.

Dosya yerelde: `C:\Users\Tunahan\Desktop\myeloma\_work\chunks\chunks.json`

In [ ]:
# 3A) Doğrudan yükleme. Beklenen: 'yüklendi: chunks.json'.
from google.colab import files
yuklenen = files.upload()
CHUNKS_YOL = list(yuklenen.keys())[0]
print('yüklendi:', CHUNKS_YOL)

In [ ]:
# 3B) Alternatif: Drive'dan. A'yı kullandıysan bu hücreyi ÇALIŞTIRMA.
# from google.colab import drive
# drive.mount('/content/drive')
# CHUNKS_YOL = '/content/drive/MyDrive/chunks.json'

In [ ]:
%%writefile embed_lib.py
"""Gomme metni kurgusu - yereldeki _scripts/embed_chunks.py ile AYNI olmali."""

AZAMI_UZUNLUK = 1024      # chunk'larin %90'i 787 token; 8192 gereksiz


def gomulecek_metin(c):
    """Gomme girdisi = baslik baglami + icerik.

    Ciplak icerik yetmiyor: bir dozaj tablosu chunk'inin kendi metninde hangi
    ilaca ait oldugu yazmayabiliyor. Belge etiketi ve bolum basligi eklenince
    chunk kendi basina anlamli hale geliyor.
    """
    m = c["metadata"]
    bas = " / ".join(str(x) for x in
                     [m.get("document_label"), m.get("section_number"),
                      m.get("section_title")] if x)
    return (bas + "\n" + c["content"]).strip()


In [ ]:
# 4) Model yükleme. İlk çalıştırmada ~2,3 GB indirir, 2-4 dakika.
#    Beklenen: 'boyut 1024, cihaz cuda'.
import json, time
from sentence_transformers import SentenceTransformer
from embed_lib import gomulecek_metin, AZAMI_UZUNLUK

chunks = json.load(open(CHUNKS_YOL, encoding='utf-8'))
metinler = [gomulecek_metin(c) for c in chunks]
print(f'{len(chunks)} chunk okundu')

t0 = time.time()
model = SentenceTransformer('BAAI/bge-m3', device='cuda')
model.max_seq_length = AZAMI_UZUNLUK
print(f'model yüklendi {time.time()-t0:.0f}sn, '
      f'boyut {model.get_sentence_embedding_dimension()}, cihaz cuda')

In [ ]:
# 5) Asıl gömme. T4'te 15-25 dakika bekleniyor; ilerleme çubuğu görürsün.
#    batch_size bellek hatası verirse 32'ye, gerekirse 16'ya düşür.
#    normalize_embeddings=True ZORUNLU - yereldeki sorgu tarafı da normalize
#    ediyor; biri normalize edilmezse kosinüs benzerliği bozulur.
import numpy as np

t1 = time.time()
vek = model.encode(metinler, batch_size=64, show_progress_bar=True,
                   normalize_embeddings=True, convert_to_numpy=True)
sure = time.time() - t1
print(f'{vek.shape[0]} vektör x {vek.shape[1]} boyut, {sure:.0f}sn '
      f'({vek.shape[0]/sure:.1f} chunk/sn)')

In [ ]:
# 6) Kaydet ve indir. İki dosya iner: .npy (vektörler) ve .meta.json.
#    altkume_idx=None demek 'tam arşiv, satır sırası chunks.json ile aynı'.
np.save('BAAI_bge-m3.npy', vek.astype('float32'))
json.dump(dict(model='BAAI/bge-m3', boyut=int(vek.shape[1]),
               chunk=int(vek.shape[0]), azami_uzunluk=AZAMI_UZUNLUK,
               saniye=round(sure, 1), altkume_idx=None,
               tarih=time.strftime('%Y-%m-%d %H:%M:%S')),
          open('BAAI_bge-m3.meta.json', 'w', encoding='utf-8'),
          indent=1, ensure_ascii=False)

from google.colab import files
files.download('BAAI_bge-m3.npy')
files.download('BAAI_bge-m3.meta.json')

## Bittikten sonra

İnen iki dosyayı şuraya koy:

```
C:\Users\Tunahan\Desktop\myeloma\_work\embeddings\
```

Orada alt küme ölçümünden kalan `BAAI_bge-m3.altkume.npy` ve
`.altkume.meta.json` var — **onları silme**, o ölçümün kaydı onlar.
Değerlendirici alt kümeyi önce arıyor, o yüzden tam arşiv ölçümüne geçerken
alt küme dosyalarının yolunu değiştirmek gerekecek; bunu ben yaparım.

Sonra bana "gömme bitti" de; tam arşivde üç modu (BM25, dense, hibrit) ölçüp
gerçek skorları çıkarayım.